# LangGraph와 AgentCore 메모리 체크포인터 (단기 메모리)

## 소개
이 노트북은 **AgentCoreMemorySaver** 체크포인터를 사용하여 Amazon Bedrock AgentCore 메모리 기능을 LangGraph와 통합하는 방법을 시연합니다. 자동 상태 체크포인팅을 통해 대화 턴 간에 **단기 메모리** 지속성에 초점을 맞추어, 에이전트가 실행 중인 맥락을 유지하고 이전 계산을 기반으로 구축할 수 있도록 합니다.

## 튜토리얼 세부 정보

| 정보                | 세부 사항                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형                                                                      |
| 에이전트 사용 사례  | 다단계 수학 계산                                                                 |
| 에이전트 프레임워크 | Langgraph                                                                        |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, Langgraph 체크포인터, 수학 도구                           |
| 예제 복잡도         | 초급                                                                             |

학습 내용:
- 자동 상태 지속성을 위해 AgentCore 메모리로 메모리 체크포인터 생성
- AgentCore 메모리 백엔드로 LangGraph의 내장 체크포인팅 시스템 사용
- 여러 상호작용 간 대화 맥락 유지
- 대화 상태 및 이력 검사 및 관리

### 시나리오 맥락

이 예제에서는 다단계 수학 계산을 수행할 수 있는 "**수학 에이전트**"를 만듭니다. 단순한 일회성 상호작용과 달리, 이 에이전트는 AgentCore 메모리의 체크포인팅 기능을 사용하여 실행 중인 맥락을 유지하므로 이전 계산을 기반으로 구축하고 여러 턴에 걸쳐 대화 흐름을 기억할 수 있습니다.

## 아키텍처
<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근 권한

### 통합 작동 방식

LangGraph와 AgentCore 메모리 간의 통합은 다음을 포함합니다:

1. LangGraph 상태 지속성을 위한 체크포인터 백엔드로 AgentCore 메모리 사용
2. 각 단계에서 대화 상태의 자동 저장 및 로드
3. 여러 동시 세션 및 액터 지원

이 접근 방식은 수동 메모리 작업 없이 원활한 상태 관리를 제공하여 더 유지보수 가능하고 확장 가능한 에이전트 아키텍처를 만듭니다.

In [ ]:
# 필요한 라이브러리 설치
!pip install -qr requirements.txt

In [ ]:
# LangGraph 및 LangChain 컴포넌트 임포트
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

In [ ]:
# 체크포인터로 사용할 AgentCoreMemorySaver 임포트
import os
import logging

from langgraph_checkpoint_aws import AgentCoreMemorySaver
from bedrock_agentcore.memory import MemoryClient

region = os.getenv('AWS_REGION', 'us-west-2')
logging.getLogger("math-agent").setLevel(logging.DEBUG)

# 메모리 리소스 생성 또는 가져오기
memory_name = "MathLanggraphAgent"
client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory['id'] # 나중에 사용할 메모리 ID 저장

### AgentCore 메모리 구성

이제 AgentCore 메모리 체크포인터를 구성하고 LLM을 초기화합니다:

- `memory_id`는 체크포인트가 저장될 AgentCore 메모리 리소스에 해당합니다
- `region`은 리소스의 AWS 리전을 지정합니다
- `MODEL_ID`는 LangGraph 에이전트를 구동할 Bedrock 모델을 정의합니다

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# 상태 지속성을 위한 체크포인터 초기화
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

# LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

### 수학 도구

에이전트가 사용할 수학 도구를 정의합니다. 이 시연에서는 두 가지 간단한 연산을 제공합니다:

In [ ]:
@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b


@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply]

### LangGraph 에이전트 구현

이제 AgentCore 메모리 체크포인터를 사용하여 LangGraph의 `create_react_agent` 빌더로 에이전트를 생성합니다:

In [ ]:
graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## 4단계: LangGraph 에이전트 실행
이제 AgentCore 메모리 체크포인터 통합으로 에이전트를 실행할 수 있습니다.

### 구성 설정
LangGraph에서 config는 호출 시 필요한 속성(예: 사용자 ID 또는 세션 ID)을 포함하는 `RuntimeConfig`입니다. 추가 문서는 여기에서 확인할 수 있습니다: [https://langchain-ai.github.io/langgraphjs/how-tos/configuration/](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)

AgentCore 메모리 체크포인터(`AgentCoreMemorySaver`)의 경우 다음을 지정해야 합니다:
- `thread_id`: AgentCore session_id에 매핑됩니다 (고유한 대화 스레드)
- `actor_id`: AgentCore actor_id에 매핑됩니다 (사용자, 에이전트 또는 기타 식별자)

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-1", # 필수: 내부적으로 Bedrock AgentCore session_id에 매핑됩니다
        "actor_id": "react-agent-1", # 필수: 내부적으로 Bedrock AgentCore actor_id에 매핑됩니다
    }
}

# "1337 곱하기 515321은 얼마인가요? 그런 다음 412를 더해서 값을 알려주세요."라는 의미의 입력
inputs = {"messages": [{"role": "user", "content": "What is 1337 times 515321? Then add 412 and return the value to me."}]}

#### 축하합니다! 에이전트가 준비되었습니다!!

### 에이전트 테스트

첫 번째 계산을 실행하여 에이전트의 동작을 확인합니다:

In [ ]:
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### 에이전트 상태 검사

AgentCore 메모리에 저장된 현재 대화 상태를 살펴봅니다. 체크포인터가 자동으로 액터 및 세션의 상태를 저장하고 검색합니다:

In [ ]:
for message in graph.get_state(config).values.get("messages"):
    print(f"{message.type}: {message.text()}")
    print("=========================================")

### 체크포인트 이력 보기

체크포인트 이력을 살펴보고 실행 중에 에이전트의 상태가 어떻게 변화했는지 확인합니다. 체크포인트는 역순으로 나열됩니다 (가장 최근 것이 먼저 표시됩니다).

In [ ]:
for checkpoint in graph.get_state_history(config):
    print(
        f"(Checkpoint ID: {checkpoint.config['configurable']['checkpoint_id']}) # of messages in state: {len(checkpoint.values.get('messages'))}"
    )

### 메모리 지속성 테스트

이제 대화를 계속하여 체크포인터의 성능을 테스트합니다. 에이전트는 이전 계산을 기억해야 합니다:

In [ ]:
# "처음에 요청한 계산이 무엇이었나요?"라는 의미의 입력
inputs = {"messages": [{"role": "user", "content": "What were the first calculations I asked you to do?"}]}

for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### 새 세션 시작

새 대화 스레드를 생성하여 세션 격리를 시연합니다. 이 새 세션에서 에이전트는 이전 계산을 기억하지 못합니다:

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2", # 새 세션 ID
        "actor_id": "react-agent-1", # 동일한 액터 ID
    }
}

# "곱하고 더하라고 요청한 값이 무엇이었나요?"라는 의미의 입력
inputs = {"messages": [{"role": "user", "content": "What values did I ask you to multiply and add?"}]}
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

## 요약

이 노트북에서 시연한 내용:

1. 체크포인팅을 위한 AgentCore 메모리 리소스 생성 방법
2. 자동 상태 지속성을 갖춘 LangGraph 에이전트 구축
3. 다단계 계산을 위한 수학 도구 구현
4. AgentCoreMemorySaver를 체크포인터 백엔드로 사용
5. 메모리 지속성 및 세션 격리 테스트

이 통합은 LangGraph의 구조화된 워크플로우와 AgentCore 메모리의 강력한 체크포인팅 기능을 결합하여 여러 상호작용에 걸쳐 맥락을 유지할 수 있는 상태 유지형 영구 AI 에이전트를 만드는 강력함을 보여줍니다.

시연된 접근 방식은 다중 에이전트 시스템, 장기 실행 워크플로우, 대화 맥락에 기반한 특화된 상태 관리를 포함한 더 복잡한 사용 사례로 확장할 수 있습니다.

### 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다.

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)